# Task 1 — Smart Restaurant Rating Predictor
**Intern:** Abhishek | **Enrollment:** CTI/A1/C358755
**Organization:** Cognifyz Technologies | **Domain:** Machine Learning

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("../Dataset/Dataset.csv")
print(f"Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Column types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("Target variable statistics:")
print(df["Aggregate rating"].describe())
not_rated = (df["Aggregate rating"] == 0).sum()
print(f"\nUnrated rows (0.0): {not_rated}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["Aggregate rating"], bins=25, color="steelblue", edgecolor="white")
ax.set_title("Distribution of Aggregate Ratings")
ax.set_xlabel("Rating")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Remove unrated restaurants
df = df[df["Aggregate rating"] > 0].copy()
print(f"After removing unrated rows: {df.shape}")

# Fill missing cuisines
df["Cuisines"] = df["Cuisines"].fillna("Unknown")

# Encode binary columns
le = LabelEncoder()
df["Has Table booking"] = le.fit_transform(df["Has Table booking"])
df["Has Online delivery"] = le.fit_transform(df["Has Online delivery"])
df["Is delivering now"] = le.fit_transform(df["Is delivering now"])

# Extract and encode primary cuisine
df["Primary Cuisine"] = df["Cuisines"].apply(lambda x: x.split(",")[0].strip())
df["Cuisine Encoded"] = le.fit_transform(df["Primary Cuisine"])
df["City Encoded"] = le.fit_transform(df["City"])

print("Encoding complete.")

## 5. Feature Selection & Split

In [ ]:
FEATURES = ["Has Table booking", "Has Online delivery", "Price range",
            "Average Cost for two", "Votes", "Country Code",
            "Cuisine Encoded", "City Encoded"]
TARGET = "Aggregate rating"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 6. Model Training & Evaluation

In [ ]:
models = {
    "Linear Regression":   LinearRegression(),
    "Decision Tree":       DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest":       RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting":   GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        "MSE": round(mean_squared_error(y_test, preds), 4),
        "MAE": round(mean_absolute_error(y_test, preds), 4),
        "R2":  round(r2_score(y_test, preds), 4),
        "preds": preds, "model": model
    }
    print(f"{name}: R2={results[name]['R2']}, MAE={results[name]['MAE']}, MSE={results[name]['MSE']}")

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame({k: {m: results[k][m] for m in ["MSE","MAE","R2"]} for k in results}).T
print(results_df)
best_name = results_df["R2"].idxmax()
print(f"\nBest Model: {best_name} (R2={results_df.loc[best_name, 'R2']})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric, color in zip(axes, ["MSE","MAE","R2"], ["#e07b54","#5b8db8","#6bbf84"]):
    results_df[metric].plot(kind="bar", ax=ax, color=color, edgecolor="white", rot=25)
    ax.set_title(metric, fontweight="bold")
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
rf = results["Random Forest"]["model"]
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
importances.plot(kind="barh", figsize=(8,5), color="teal", edgecolor="white")
plt.title("Feature Importance — Random Forest", fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Conclusions

**Best Model:** Gradient Boosting (R² ≈ 0.6246)

**Key Findings:**
- Votes, Price range, and Average Cost for two are the strongest rating predictors
- Tree-based ensembles outperform linear models due to non-linear patterns
- Rating text and Rating color were excluded to avoid data leakage
- ~62% of rating variance is explained by available features